# Project 1: Is Stack Overflow Dying?
## What the decline tells us about how developers work in the AI era

Stack Overflow launched in 2008 and became the primary place developers went for help for over a decade. Then in November 2022, ChatGPT launched. This notebook uses monthly post-count data across 14 programming languages from 2008 to present to show what happened next.

**Questions this notebook answers:**
1. When did the decline start — and how severe is it?
2. Was ChatGPT the cause, or an accelerant of a trend already in motion?
3. Which languages were hit hardest, and which held up?
4. Is Python's resilience genuine, or statistical noise?

**Where this leads:** The findings here showed that Stack Overflow post volume is no longer a reliable signal for language relevance after 2022 — making it an unsuitable primary data source for workforce strategy. That finding directly motivated [Project 2: Language Market Index (LMI)](proj_2_language_market_index.ipynb), which replaces SO volume with a composite index built from four independent sources.

## 1. Setup

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.patches as mpatches

df = pd.read_csv('../data/raw/so/QueryResults.csv')
df.columns = ['DATE', 'TAG', 'POSTS']
df['DATE'] = pd.to_datetime(df['DATE'])

pivot = df.pivot(index='DATE', columns='TAG', values='POSTS').fillna(0)

AI_INFLECTION = pd.Timestamp('2022-11-01')

print(f"Date range: {df['DATE'].min().date()} → {df['DATE'].max().date()}")
print(f"Languages tracked: {len(pivot.columns)}")

os.makedirs('../plots/so', exist_ok=True)


## 2. The Overall Decline

Total monthly posts across all tracked languages — the clearest view of platform-level activity.

In [ ]:
total_monthly = pivot.sum(axis=1)
smoothed_total = total_monthly.rolling(window=6).mean()

peak_date  = smoothed_total.idxmax()
peak_value = smoothed_total.max()
latest_value = smoothed_total.dropna().iloc[-1]
drop_pct = (latest_value - peak_value) / peak_value * 100

print(f"Peak activity:   {peak_date.strftime('%B %Y')} — {peak_value:,.0f} posts/month")
print(f"Latest reading:  {smoothed_total.dropna().index[-1].strftime('%B %Y')} — {latest_value:,.0f} posts/month")
print(f"Drop from peak:  {drop_pct:.1f}%")

In [ ]:
fig, ax = plt.subplots(figsize=(16, 7))

ax.fill_between(smoothed_total.index, smoothed_total, alpha=0.15, color='#5c6cfa')
ax.plot(smoothed_total.index, smoothed_total, color='#5c6cfa', linewidth=2.5)

# Peak annotation
ax.annotate(
    f'Peak: {peak_value:,.0f}/month\n{peak_date.strftime("%b %Y")}',
    xy=(peak_date, peak_value),
    xytext=(peak_date - pd.DateOffset(months=30), peak_value * 0.92),
    arrowprops=dict(arrowstyle='->', color='white', lw=1.2),
    fontsize=11, color='white'
)

# ChatGPT line
ax.axvline(AI_INFLECTION, color='#e74c3c', linewidth=1.5, linestyle='--', alpha=0.8)
ax.text(AI_INFLECTION + pd.DateOffset(months=1), peak_value * 0.6,
        'ChatGPT launch\nNov 2022', color='#e74c3c', fontsize=11)

ax.set_title('Total Stack Overflow Posts Across All Tracked Languages (6-month rolling avg)',
             fontsize=15, fontweight='bold', pad=14)
ax.set_xlabel('Date', fontsize=12)
ax.set_ylabel('Total Posts per Month', fontsize=12)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
ax.tick_params(labelsize=11)
ax.set_facecolor('#0f1117')
fig.patch.set_facecolor('#0f1117')
ax.tick_params(colors='#c0c0c0')
ax.xaxis.label.set_color('#c0c0c0')
ax.yaxis.label.set_color('#c0c0c0')
ax.title.set_color('white')
for spine in ax.spines.values():
    spine.set_edgecolor('#2d3148')
plt.tight_layout()
plt.savefig('../plots/so_decline_analysis/total_monthly_posts_decline.png', dpi=150, facecolor=fig.get_facecolor())
plt.show()

In [ ]:
smoothed_pivot = pivot.rolling(window=6).mean()

fig, ax = plt.subplots(figsize=(16, 8))
for lang in pivot.columns:
    ax.plot(smoothed_pivot.index, smoothed_pivot[lang], linewidth=1.5, label=lang, alpha=0.85)

ax.axvline(AI_INFLECTION, color='#e74c3c', linewidth=1.5, linestyle='--', alpha=0.8)
ax.text(AI_INFLECTION + pd.DateOffset(months=2),
        smoothed_pivot.max().max() * 0.88,
        'ChatGPT\nNov 2022', color='#e74c3c', fontsize=10)
ax.set_title('Stack Overflow Monthly Posts by Language (6-month rolling avg)',
             fontsize=14, fontweight='bold', pad=14)
ax.set_xlabel('Date', fontsize=12)
ax.set_ylabel('Posts per Month', fontsize=12)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
ax.legend(fontsize=9, loc='upper right', ncol=2, framealpha=0.3,
          labelcolor='#c0c0c0', facecolor='#1a1d27', edgecolor='#2d3148')
ax.set_facecolor('#0f1117')
fig.patch.set_facecolor('#0f1117')
ax.tick_params(colors='#c0c0c0', labelsize=10)
ax.xaxis.label.set_color('#c0c0c0')
ax.yaxis.label.set_color('#c0c0c0')
ax.title.set_color('white')
for spine in ax.spines.values():
    spine.set_edgecolor('#2d3148')
plt.tight_layout()
plt.savefig('../plots/so_decline_analysis/monthly_posts_per_language.png', dpi=150, facecolor=fig.get_facecolor())
plt.show()

## 3. Pre vs Post AI: Which Languages Were Hit Hardest?

Comparing average monthly posts in the 24 months before ChatGPT's launch vs the 24 months after. Some languages may have held up better — which would itself be a signal.

In [ ]:
pre_ai  = pivot[pivot.index <  AI_INFLECTION].iloc[-24:].mean()
post_ai = pivot[pivot.index >= AI_INFLECTION].iloc[:24].mean()

ai_table = pd.DataFrame({
    'Pre-AI avg/month':  pre_ai.round(0).astype(int),
    'Post-AI avg/month': post_ai.round(0).astype(int),
    'Drop (%)':          ((post_ai - pre_ai) / pre_ai * 100).round(1),
}).sort_values('Drop (%)')

print('=== Impact of ChatGPT launch on monthly post volume ===')
print(ai_table.to_string())
print(f'\nAverage drop across all languages: {ai_table["Drop (%)"].mean():.1f}%')

In [ ]:
drops = ai_table['Drop (%)'].sort_values()
colors = ['#e74c3c' if v < -50 else '#f39c12' if v < -30 else '#2ecc71' for v in drops]

fig, ax = plt.subplots(figsize=(12, 7))
bars = ax.barh(drops.index, drops.values, color=colors, edgecolor='none')

for bar, val in zip(bars, drops.values):
    ax.text(val - 0.5, bar.get_y() + bar.get_height() / 2,
            f'{val:.1f}%', va='center', ha='right', fontsize=10, color='white')

ax.axvline(0, color='white', linewidth=0.6)
ax.axvline(drops.mean(), color='#f39c12', linewidth=1, linestyle=':', alpha=0.8)
ax.text(drops.mean() + 0.3, ax.get_ylim()[0] + 0.3,
        f'avg {drops.mean():.1f}%', color='#f39c12', fontsize=9)

ax.set_title('Post-volume drop per language: 24 months before vs after ChatGPT launch',
             fontsize=13, fontweight='bold', pad=12)
ax.set_xlabel('Change in average monthly posts (%)', fontsize=11)

patches = [
    mpatches.Patch(color='#e74c3c', label='> 50% drop'),
    mpatches.Patch(color='#f39c12', label='30–50% drop'),
    mpatches.Patch(color='#2ecc71', label='< 30% drop'),
]
ax.legend(handles=patches, fontsize=10)
ax.set_facecolor('#0f1117')
fig.patch.set_facecolor('#0f1117')
ax.tick_params(colors='#c0c0c0', labelsize=11)
ax.xaxis.label.set_color('#c0c0c0')
ax.title.set_color('white')
for spine in ax.spines.values():
    spine.set_edgecolor('#2d3148')
plt.tight_layout()
plt.savefig('../plots/so_decline_analysis/pre_post_chatgpt_drop_per_language.png', dpi=150, facecolor=fig.get_facecolor())
plt.show()

## 4. Velocity Analysis: Cause or Accelerant?

The pre/post comparison shows a clear drop. But it doesn't answer whether ChatGPT *caused* the decline or whether Stack Overflow was already losing momentum and ChatGPT simply sped it up.

To answer this, we look at the month-over-month rate of change in total posts — the *velocity* of the platform — and check whether that velocity was already negative before November 2022.

In [ ]:
velocity = total_monthly.pct_change() * 100
smoothed_vel = velocity.rolling(window=6).mean().dropna()

# Trim early explosive-growth era (2008-2012) — those spikes dominate the y-axis
smoothed_vel = smoothed_vel[smoothed_vel.index >= pd.Timestamp('2013-01-01')]

recent_pre = pd.Timestamp('2019-11-01')
pre_avg  = float(smoothed_vel[(smoothed_vel.index >= recent_pre) & (smoothed_vel.index < AI_INFLECTION)].mean())
post_avg = float(smoothed_vel[smoothed_vel.index >= AI_INFLECTION].mean())

print(f'Pre-AI monthly change (2019–2022 avg):  {pre_avg:.2f}%/month')
print(f'Post-AI monthly change (post Nov 2022): {post_avg:.2f}%/month')
print(f'Acceleration factor: {post_avg / pre_avg:.1f}×')

fig, ax = plt.subplots(figsize=(16, 6))

bar_colors = ['#f07070' if v < 0 else '#4ecb71' for v in smoothed_vel]
ax.bar(smoothed_vel.index, smoothed_vel.values, color=bar_colors, width=20, edgecolor='none', alpha=0.85)

ax.axhline(0, color='#8b90a8', linewidth=0.8)
ax.axvline(AI_INFLECTION, color='#e74c3c', linewidth=1.5, linestyle='--', alpha=0.8)
ax.text(AI_INFLECTION + pd.DateOffset(months=1), smoothed_vel.max() * 0.8,
        'ChatGPT\nNov 2022', color='#e74c3c', fontsize=10)

ax.axhline(pre_avg, color='#4ecb71', linewidth=1.2, linestyle=':', alpha=0.8)
ax.text(smoothed_vel.index[2], pre_avg + 0.3,
        f'pre-AI avg {pre_avg:.1f}%/mo', color='#4ecb71', fontsize=9)

ax.axhline(post_avg, color='#f07070', linewidth=1.2, linestyle=':', alpha=0.8)
ax.text(smoothed_vel.index[-40], post_avg - 0.8,
        f'post-AI avg {post_avg:.1f}%/mo', color='#f07070', fontsize=9)

ax.set_title('Stack Overflow Velocity: Month-over-Month Change (6-month rolling avg, 2013+)',
             fontsize=13, fontweight='bold', pad=12)
ax.set_xlabel('Date', fontsize=11)
ax.set_ylabel('Monthly change (%)', fontsize=11)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:.1f}%'))
ax.set_facecolor('#0f1117')
fig.patch.set_facecolor('#0f1117')
ax.tick_params(colors='#c0c0c0', labelsize=10)
ax.xaxis.label.set_color('#c0c0c0')
ax.yaxis.label.set_color('#c0c0c0')
ax.title.set_color('white')
for spine in ax.spines.values():
    spine.set_edgecolor('#2d3148')
plt.tight_layout()
plt.savefig('../plots/so_decline_analysis/month_over_month_velocity.png', dpi=150, facecolor=fig.get_facecolor())
plt.show()

## 5. Share of Activity Over Time

If we strip out the platform-level decline and look at each language's share of monthly posts, we can see which languages are actually growing or shrinking *relative to each other* — independent of what Stack Overflow is doing overall.

In [ ]:
row_totals = pivot.sum(axis=1).replace(0, np.nan)
share = pivot.div(row_totals, axis=0) * 100
smoothed_share = share.rolling(window=6).mean()

# Focus on the languages with meaningful share
top_langs = share.mean().sort_values(ascending=False).head(8).index

fig, ax = plt.subplots(figsize=(16, 8))

for lang in top_langs:
    ax.plot(smoothed_share.index, smoothed_share[lang], linewidth=2, label=lang)

ax.axvline(AI_INFLECTION, color='white', linewidth=1.2, linestyle='--', alpha=0.6)
ax.text(AI_INFLECTION + pd.DateOffset(months=1),
        smoothed_share[top_langs].max().max() * 0.95,
        'ChatGPT\nNov 2022', color='white', fontsize=10, alpha=0.8)

ax.set_title('Language Share of Monthly Stack Overflow Posts (6-month rolling avg)',
             fontsize=14, fontweight='bold', pad=14)
ax.set_xlabel('Date', fontsize=12)
ax.set_ylabel('Share of monthly posts (%)', fontsize=12)
ax.legend(fontsize=11, loc='upper left', ncol=2)
ax.set_facecolor('#0f1117')
fig.patch.set_facecolor('#0f1117')
ax.tick_params(colors='#c0c0c0', labelsize=11)
ax.xaxis.label.set_color('#c0c0c0')
ax.yaxis.label.set_color('#c0c0c0')
ax.title.set_color('white')
for spine in ax.spines.values():
    spine.set_edgecolor('#2d3148')
plt.tight_layout()
plt.savefig('../plots/so_decline_analysis/language_share_over_time.png', dpi=150, facecolor=fig.get_facecolor())
plt.show()

## 6. Language-Level Momentum

The platform-level view shows a universal decline. Drilling down to per-language momentum —
comparing the most recent 24 months against the prior 24 months — shows which languages are
growing or shrinking *relative to their own history*, independent of the platform decline.

In [ ]:
recent   = pivot.iloc[-24:].mean()
prior    = pivot.iloc[-48:-24].mean()
momentum = ((recent - prior) / prior * 100).round(1).sort_values()

print('Language momentum (recent 24 mo vs prior 24 mo):')
for lang, val in momentum.items():
    print(f'  {lang:12s}  {val:+.1f}%')

In [ ]:
bar_colors = ['#f07070' if v < 0 else '#4ecb71' for v in momentum]

fig, ax = plt.subplots(figsize=(12, 8))
bars = ax.barh(momentum.index, momentum.values, color=bar_colors, edgecolor='none')

for bar, val in zip(bars, momentum.values):
    label = f'{val:+.1f}%'
    x_pos = val - 0.8 if val < 0 else val + 0.5
    ha = 'right' if val < 0 else 'left'
    ax.text(x_pos, bar.get_y() + bar.get_height() / 2,
            label, va='center', ha=ha, fontsize=10, color='white')

ax.axvline(0, color='#8b90a8', linewidth=0.8)
ax.set_title('Language Momentum Score: 24-month growth vs prior 24 months',
             fontsize=13, fontweight='bold', pad=12)
ax.set_xlabel('Momentum (%)', fontsize=11)
ax.set_facecolor('#0f1117')
fig.patch.set_facecolor('#0f1117')
ax.tick_params(colors='#c0c0c0', labelsize=11)
ax.xaxis.label.set_color('#c0c0c0')
ax.title.set_color('white')
for spine in ax.spines.values():
    spine.set_edgecolor('#2d3148')
plt.tight_layout()
plt.savefig('../plots/so_decline_analysis/language_momentum_score.png', dpi=150, facecolor=fig.get_facecolor())
plt.show()

In [ ]:
total_posts = pivot.sum()

LC_COLORS = {
    'Dominant':  '#5bc0f8',
    'Rising':    '#4ecb71',
    'Mature':    '#f0c040',
    'Declining': '#f07070',
    'Niche':     '#aaaaaa',
}

def classify_so(total, mom):
    if total > 1_500_000 and mom >= 5:
        return 'Dominant'
    if mom > 20:
        return 'Rising'
    if mom < -20:
        return 'Declining'
    if total > 400_000:
        return 'Mature'
    return 'Niche'

lifecycle = {lang: classify_so(total_posts[lang], momentum[lang]) for lang in total_posts.index}

fig, ax = plt.subplots(figsize=(13, 9))
for lang in total_posts.index:
    stage = lifecycle[lang]
    ax.scatter(total_posts[lang], momentum[lang], color=LC_COLORS[stage], s=160, zorder=5)
    ax.annotate(lang, (total_posts[lang], momentum[lang]),
                xytext=(8, 0), textcoords='offset points', fontsize=10, color='#c0c0c0')

ax.axhline(0,  color='#8b90a8', linewidth=1,   linestyle='--', alpha=0.6)
ax.axhline(20, color='#8b90a8', linewidth=0.7, linestyle=':',  alpha=0.4)
ax.axvline(500_000, color='#8b90a8', linewidth=0.7, linestyle=':', alpha=0.4)

patches = [mpatches.Patch(color=c, label=s) for s, c in LC_COLORS.items()]
ax.legend(handles=patches, title='Lifecycle Stage', fontsize=10, title_fontsize=10,
          facecolor='#1a1d27', edgecolor='#2d3148', labelcolor='#c0c0c0')
ax.set_title('Language Lifecycle Matrix: Volume vs Momentum',
             fontsize=13, fontweight='bold', pad=12)
ax.set_xlabel('Total Stack Overflow Posts (all time)', fontsize=11)
ax.set_ylabel('Momentum Score (%)', fontsize=11)
ax.xaxis.set_major_formatter(mticker.FuncFormatter(
    lambda x, _: f'{x/1_000_000:.1f}M' if x >= 1_000_000 else f'{int(x/1000)}K'))
ax.set_facecolor('#0f1117')
fig.patch.set_facecolor('#0f1117')
ax.tick_params(colors='#c0c0c0', labelsize=10)
ax.xaxis.label.set_color('#c0c0c0')
ax.yaxis.label.set_color('#c0c0c0')
ax.title.set_color('white')
for spine in ax.spines.values():
    spine.set_edgecolor('#2d3148')
plt.tight_layout()
plt.savefig('../plots/so_decline_analysis/lifecycle_matrix_volume_vs_momentum.png', dpi=150, facecolor=fig.get_facecolor())
plt.show()

## 7. Does Stack Overflow Momentum Predict Job Demand?

If SO momentum were a reliable workforce signal, languages gaining SO share should also
dominate job postings. The chart below tests that directly — SO momentum vs Adzuna job counts.
A negative correlation means the signal is not just noisy but *inverted*.

In [ ]:
import os
from scipy import stats

NORM_PATH = '../data/processed/normalized.csv'

if os.path.exists(NORM_PATH):
    norm   = pd.read_csv(NORM_PATH)
    adzuna = norm[norm['source'] == 'adzuna_total'].set_index('language')['raw_value']

    common = [l for l in momentum.index if l in adzuna.index]
    x      = momentum[common].values
    y      = adzuna[common].values

    slope, intercept, r, p, _ = stats.linregress(x, y)
    x_line = np.linspace(x.min(), x.max(), 100)
    y_line = slope * x_line + intercept

    fig, ax = plt.subplots(figsize=(12, 8))
    ax.plot(x_line, y_line, color='#e74c3c', linewidth=1.5, linestyle='--', alpha=0.7)
    ax.scatter(x, y, color='#5c6cfa', s=130, zorder=5)
    for lang, xi, yi in zip(common, x, y):
        ax.annotate(lang, (xi, yi), xytext=(8, 0),
                    textcoords='offset points', fontsize=10, color='#c0c0c0')

    ax.set_title(f'Stack Overflow Momentum vs Job Market Demand\n(Pearson r = {r:.2f})',
                 fontsize=13, fontweight='bold', pad=12)
    ax.set_xlabel('Momentum Score (%)', fontsize=11)
    ax.set_ylabel('Job Postings (Adzuna)', fontsize=11)
    ax.set_facecolor('#0f1117')
    fig.patch.set_facecolor('#0f1117')
    ax.tick_params(colors='#c0c0c0', labelsize=10)
    ax.xaxis.label.set_color('#c0c0c0')
    ax.yaxis.label.set_color('#c0c0c0')
    ax.title.set_color('white')
    for spine in ax.spines.values():
        spine.set_edgecolor('#2d3148')
    plt.tight_layout()
    plt.savefig('../plots/so_decline_analysis/so_momentum_vs_job_demand.png', dpi=150, facecolor=fig.get_facecolor())
    plt.show()

    print(f'Pearson r = {r:.2f}  (p = {p:.3f})')
    print('Negative correlation: languages declining on SO still dominate the job market.')
    print('SO momentum is not just noisy — it is an inverted signal for workforce demand.')
else:
    print(f'Adzuna data not found at {NORM_PATH}')
    print('Run the LMI pipeline first: cd lmi && python run_pipeline.py')

## 8. Findings

**1. Stack Overflow is in structural decline**

Total post volume dropped **~54%** on average across all languages in the 24 months after ChatGPT launched vs the 24 months before. This is not a blip — it maps directly to developers switching to AI tools for answers they previously searched Stack Overflow for.

**2. ChatGPT was an accelerant, not the cause**

SO was already declining at **−0.5%/month** in the three years before ChatGPT launched (2019–2022). ChatGPT made that decline **15× steeper** (−7.6%/month post-launch). The platform was already losing ground to alternative resources — AI simply collapsed the timeline.

**3. The decline is uneven — and that's the signal**

JavaScript lost the most relative share (−16.2%). Go (+44.2%) and C# (+30.9%) actually *gained* share. Languages that gained relative share are likely those where community problem-solving remains essential and AI tools are less capable — enterprise and systems ecosystems with more complex, context-heavy questions.

**4. Python users shifted to AI faster than anyone**

Python's share of SO posts declined **−4.6%** after the AI inflection point — meaning Python dropped more than average in relative terms (−60% absolute vs −54% average). Python developers were among the earliest adopters of AI-assisted coding, which makes sense: Python is the primary language of AI tooling itself.

---

**Implication for workforce strategy**

Stack Overflow is no longer a reliable absolute signal for developer interest. The languages showing the steepest SO decline (JavaScript, Python) are not declining — their developers simply moved to AI tools first. This inversion makes raw SO counts actively misleading for hiring decisions. That finding motivated building the Language Market Index (LMI). See [Project 2](proj_2_language_market_index.ipynb).